# Table 2 - Explanation Quality on Benchmark Datasets

Produces the **CF-Greedy (Ours)** row of Table 2: precision (`pre`), explanation size
(`average number of exps`, i.e. the Minimum Information Perturbation) and runtime on
BA-Shapes, Tree-Cycles and Mutag_0.

To keep the comparison against GNNExplainer / CF-GNNExplainer / CF^2 exactly fair, this runs
CF-Greedy **on the baselines' own pre-trained models and data splits**, taken from the CF^2
reference implementation (`gnn_cff`). Clone it next to this repository first:

```bash
git clone https://github.com/chrisjtan/gnn_cff.git
```

and follow its README to train the three models so that
`gnn_cff/log/{Mutagenicity_0,BA_Shapes,Tree_Cycles}_logs/` exist. The baseline rows of Table 2
(GNNExplainer, CF-GNNExplainer, CF^2) are produced by `gnn_cff`'s own evaluation scripts, not
by this repository.

Requires `dgl` in addition to this repository's requirements.

## Datasets & Models
| Dataset | Model | Task |
|---|---|---|
| Mutagenicity_0 | GCNGraph (graph-level) | Binary graph classification |
| BA_Shapes | GCNNodeBAShapes (node-level) | 4-class node classification |
| Tree_Cycles | GCNNodeTreeCycles (node-level) | 2-class node classification |

**Method**: direct greedy edge removal, which preserves exact edge identity (rather than
`get_local_counterfactual` + ratio-based conversion, which is lossy for uniform features).

In [ ]:
# Bootstrap: run this notebook from either the repo root or notebooks/.
import os, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print('Working directory:', os.getcwd())

In [ ]:
import sys, os
import numpy as np
import torch
import tqdm
import dgl
from dgl import load_graphs

# Path to a checkout of the CF^2 reference implementation (gnn_cff), which
# ships the pre-trained models and DGL graphs for the three benchmarks:
#     git clone https://github.com/chrisjtan/gnn_cff.git
# Override with the GNN_CFF_DIR environment variable if it lives elsewhere.
PROJECT_ROOT = os.path.abspath('.')
GNN_CFF_DIR = os.path.abspath(os.environ.get('GNN_CFF_DIR', os.path.join('..', 'gnn_cff')))
if not os.path.isdir(GNN_CFF_DIR):
    raise FileNotFoundError(
        f'gnn_cff not found at {GNN_CFF_DIR}. Clone it next to this repository '
        'or set the GNN_CFF_DIR environment variable.'
    )
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
if GNN_CFF_DIR not in sys.path:
    sys.path.insert(0, GNN_CFF_DIR)

from torch_geometric.data import Data
from models.gcn import GCNGraph, GCNNodeBAShapes, GCNNodeTreeCycles

DEVICE = 'cpu'
torch.manual_seed(0)
np.random.seed(0)
print('G2I root:', PROJECT_ROOT)
print('gnn_cff root:', GNN_CFF_DIR)

In [ ]:
# ============================================================
# Wrappers + Greedy Edge Explanation
# ============================================================

class GCNGraphWrapper(torch.nn.Module):
    """Wraps GCNGraph (graph-level DGL) for greedy search's PyG interface."""
    def __init__(self, gcn_model):
        super().__init__()
        self.gcn_model = gcn_model
    def forward(self, x, edge_index, edge_weight=None):
        n = x.shape[0]
        edge_set = set()
        for k in range(edge_index.shape[1]):
            edge_set.add((edge_index[0, k].item(), edge_index[1, k].item()))
        src_list, dst_list, weights = [], [], []
        for i in range(n):
            for j in range(n):
                src_list.append(i); dst_list.append(j)
                weights.append(1.0 if (i, j) in edge_set else 0.0)
        g = dgl.graph((src_list, dst_list), num_nodes=n)
        g.ndata['feat'] = x.float()
        e_w = torch.tensor(weights, dtype=torch.float32)
        g.edata['weight'] = e_w
        pred = self.gcn_model(g, x.float(), e_w)
        return pred.squeeze().expand(n)


class NodeModelWrapper(torch.nn.Module):
    """Wraps node-level DGL models for greedy search's PyG interface."""
    def __init__(self, gcn_model, target_node, pred_class_idx):
        super().__init__()
        self.gcn_model = gcn_model
        self.target_node = target_node
        self.pred_class_idx = pred_class_idx
    def forward(self, x, edge_index, edge_weight=None):
        n = x.shape[0]
        edge_set = set()
        for k in range(edge_index.shape[1]):
            edge_set.add((edge_index[0, k].item(), edge_index[1, k].item()))
        src_list, dst_list, weights = [], [], []
        for i in range(n):
            for j in range(n):
                src_list.append(i); dst_list.append(j)
                weights.append(1.0 if (i, j) in edge_set else 0.0)
        g = dgl.graph((src_list, dst_list), num_nodes=n)
        g.ndata['feat'] = x.float()
        e_w = torch.tensor(weights, dtype=torch.float32)
        g.edata['weight'] = e_w
        pred = self.gcn_model(g, x.float(), e_w, torch.tensor(self.target_node))
        return pred[0, self.pred_class_idx].expand(n)


def dgl_to_pyg(g):
    """Convert DGL complete graph to PyG Data (only real edges)."""
    n = g.num_nodes()
    x = g.ndata['feat'].float()
    w = g.edata['weight'].numpy()
    src, dst = [], []
    for i in range(n):
        for j in range(n):
            if w[i * n + j] > 0:
                src.append(i); dst.append(j)
    return Data(x=x, edge_index=torch.tensor([src, dst], dtype=torch.long))


def greedy_edge_explanation(data, node_idx, model, max_edges=10):
    """
    Greedy edge removal: remove edges of node_idx one by one to flip prediction.
    
    Unlike get_local_counterfactual + ratio conversion, this directly tracks
    which specific edges are removed — no information loss.
    Returns empty set if prediction doesn't flip.
    """
    ei = data.edge_index
    neighbors = list(set(ei[1, ei[0] == node_idx].tolist()))
    
    with torch.no_grad():
        orig_pred = model(data.x, ei)[0].item()
    if orig_pred < 0.5:
        return set()
    
    removed_nbrs = set()
    current_ei = ei
    
    for step in range(max_edges):
        with torch.no_grad():
            cur_pred = model(data.x, current_ei)[0].item()
        if cur_pred < 0.5:
            break  # flipped
        
        remaining = [nb for nb in neighbors if nb not in removed_nbrs]
        if not remaining:
            break
        
        best_pred, best_nbr = cur_pred, None
        for nbr in remaining:
            mask = ~(((current_ei[0]==node_idx) & (current_ei[1]==nbr)) |
                     ((current_ei[0]==nbr) & (current_ei[1]==node_idx)))
            test_ei = current_ei[:, mask]
            with torch.no_grad():
                new_pred = model(data.x, test_ei)[0].item()
            if new_pred < best_pred:
                best_pred = new_pred
                best_nbr = nbr
        
        if best_nbr is None:
            break
        removed_nbrs.add(best_nbr)
        mask = ~(((current_ei[0]==node_idx) & (current_ei[1]==best_nbr)) |
                 ((current_ei[0]==best_nbr) & (current_ei[1]==node_idx)))
        current_ei = current_ei[:, mask]
    
    # Only return edges if prediction actually flipped
    with torch.no_grad():
        final_pred = model(data.x, current_ei)[0].item()
    if final_pred >= 0.5:
        return set()
    
    removed_edges = set()
    for nbr in removed_nbrs:
        removed_edges.add((node_idx, nbr))
        removed_edges.add((nbr, node_idx))
    return removed_edges

print('Wrappers and greedy_edge_explanation defined.')

---
# 1. Mutagenicity_0 Dataset
- **Model**: GCNGraph (graph-level binary classification)
- **Selection**: `test[:200]`, correct positive predictions (same as gnn_cff)

In [ ]:
# Load Mutagenicity_0
MODEL_PATH = os.path.join(GNN_CFF_DIR, 'log', 'Mutagenicity_0_logs')
graphs, label_dict = load_graphs(os.path.join(MODEL_PATH, 'dgl_graph.bin'))
labels = label_dict['labels']
feat_dim = graphs[0].ndata['feat'].shape[1]
test_indices = np.load(os.path.join(MODEL_PATH, 'test_indices.pickle'), allow_pickle=True)

base_model = GCNGraph(feat_dim, 128).to(DEVICE)
base_model.load_state_dict(torch.load(os.path.join(MODEL_PATH, 'model.model'), map_location=DEVICE))
base_model.eval()
for param in base_model.parameters():
    param.requires_grad = False

wrapper = GCNGraphWrapper(base_model)
wrapper.eval()

correct = sum(1 for gid in test_indices
              if int(torch.round(base_model(graphs[gid], graphs[gid].ndata['feat'].float(),
                                            graphs[gid].edata['weight'])[0, 0]).item()) == int(labels[gid].item()))
print(f'Dataset: {len(graphs)} graphs, feat_dim={feat_dim}')
print(f'Test accuracy: {correct}/{len(test_indices)} = {correct/len(test_indices):.4f}')

In [ ]:
# Run greedy edge explanation on Mutagenicity_0
target_gids = [gid for gid in test_indices[:200]
               if int(torch.round(base_model(graphs[gid], graphs[gid].ndata['feat'].float(),
                                             graphs[gid].edata['weight'])[0,0]).item()) == 1
               and int(labels[gid].item()) == 1]
print(f'Target graphs: {len(target_gids)}')

exp_dict = {}
for gid in tqdm.tqdm(target_gids):
    pyg_data = dgl_to_pyg(graphs[gid])
    n = pyg_data.x.shape[0]
    best_removed, best_size = None, float('inf')
    for node_idx in range(n):
        removed = greedy_edge_explanation(pyg_data, node_idx, wrapper, max_edges=10)
        if len(removed) > 0 and len(removed) < best_size:
            best_size = len(removed)
            best_removed = removed
            if best_size <= 4:
                break
    if best_removed is not None:
        exp_dict[gid] = best_removed

print(f'Explained: {len(exp_dict)}/{len(target_gids)}')

In [ ]:
# Mutagenicity_0 metrics
pn_count = ps_count = 0
pres_list, recs_list, f1s_list, accs_list, exp_nums = [], [], [], [], []

for gid in exp_dict:
    g = graphs[gid]; removed = exp_dict[gid]
    pyg_data = dgl_to_pyg(g); n = pyg_data.x.shape[0]
    exp_nums.append(len(removed) / 2.0)
    ei = pyg_data.edge_index
    
    # PN
    pn_edges = [[ei[0,k].item(), ei[1,k].item()] for k in range(ei.shape[1])
                if (ei[0,k].item(), ei[1,k].item()) not in removed]
    pn_ei = torch.tensor(pn_edges, dtype=torch.long).t().contiguous() if pn_edges else torch.zeros((2,0), dtype=torch.long)
    with torch.no_grad():
        if wrapper(pyg_data.x, pn_ei)[0].item() < 0.5: pn_count += 1
    # PS
    ps_ei = torch.tensor(list(removed), dtype=torch.long).t().contiguous()
    with torch.no_grad():
        if wrapper(pyg_data.x, ps_ei)[0].item() > 0.5: ps_count += 1
    # Precision/Recall
    e_labels = g.edata['label'].numpy()
    exp_mask = np.zeros(n * n)
    for (s, d) in removed: exp_mask[s * n + d] = 1
    TP = sum(1 for i in range(n*n) if exp_mask[i]==1 and e_labels[i]==1)
    FP = sum(1 for i in range(n*n) if exp_mask[i]==1 and e_labels[i]!=1)
    FN = sum(1 for i in range(n*n) if exp_mask[i]!=1 and e_labels[i]==1)
    TN = n*n - TP - FP - FN
    pre = TP/(TP+FP) if TP > 0 else 0; rec = TP/(TP+FN) if TP > 0 else 0
    f1 = 2*pre*rec/(pre+rec) if (pre+rec) > 0 else 0; acc = (TP+TN)/(n*n)
    pres_list.append(pre); recs_list.append(rec); f1s_list.append(f1); accs_list.append(acc)

total = len(exp_dict)
mutag_PN = pn_count/total if total else 0; mutag_PS = ps_count/total if total else 0
mutag_PNS = 2*mutag_PN*mutag_PS/(mutag_PN+mutag_PS) if (mutag_PN+mutag_PS) > 0 else 0
mutag_avg = np.mean(exp_nums) if exp_nums else 0

print(f'average number of exps: {mutag_avg:.2f}')
print(f'PN {mutag_PN:.4f}'); print(f'PS {mutag_PS:.4f}'); print(f'PNS {mutag_PNS:.4f}')
print(f'acc:  {np.mean(accs_list):.4f}  pre:  {np.mean(pres_list):.4f}  '
      f'rec:  {np.mean(recs_list):.4f}  f1:  {np.mean(f1s_list):.4f}')

---
# 2. BA_Shapes Dataset
- **Model**: GCNNodeBAShapes (4-class node classification with softmax)
- **Selection**: Test nodes with correct non-zero class prediction
- **Metrics**: PN/PS use argmax-based flip (matching `NodeExplainerEdgeMulti`)

In [ ]:
# Load BA_Shapes
BA_PATH = os.path.join(GNN_CFF_DIR, 'log', 'BA_Shapes_logs')
ba_graphs, ba_label_dict = load_graphs(os.path.join(BA_PATH, 'dgl_graph.bin'))
ba_labels = ba_label_dict['labels']
ba_targets = np.load(os.path.join(BA_PATH, 'targets.pickle'), allow_pickle=True)
ba_test_indices = np.load(os.path.join(BA_PATH, 'test_indices.pickle'), allow_pickle=True)

ba_model = GCNNodeBAShapes(ba_graphs[0].ndata['feat'].shape[1], 16, num_classes=4,
                           device=DEVICE, if_exp=True).to(DEVICE)
ba_model.load_state_dict(torch.load(os.path.join(BA_PATH, 'model.model'), map_location=DEVICE))
ba_model.eval()
for param in ba_model.parameters():
    param.requires_grad = False

correct = sum(1 for gid in ba_test_indices
              if torch.argmax(ba_model(ba_graphs[gid], ba_graphs[gid].ndata['feat'].float(),
                                       ba_graphs[gid].edata['weight'],
                                       torch.tensor(ba_targets[gid]))[0]).item()
              == torch.argmax(ba_labels[gid]).item())
print(f'Dataset: {len(ba_graphs)} subgraphs')
print(f'Test accuracy: {correct}/{len(ba_test_indices)} = {correct/len(ba_test_indices):.4f}')

In [ ]:
import time

# Start the timer
start_time = time.perf_counter()

# Run greedy edge explanation on BA_Shapes
ba_target_gids = []
for gid in ba_test_indices:
    g = ba_graphs[gid]; target = ba_targets[gid]
    with torch.no_grad():
        pred = ba_model(g, g.ndata['feat'].float(), g.edata['weight'], torch.tensor(target))
    pred_class = torch.argmax(pred[0]).item()
    true_class = torch.argmax(ba_labels[gid]).item()
    if pred_class != 0 and true_class != 0 and pred_class == true_class:
        ba_target_gids.append((gid, pred_class))
print(f'Target nodes: {len(ba_target_gids)}')

ba_exp_dict, ba_pred_dict = {}, {}
for gid, pred_class in ba_target_gids:
    g = ba_graphs[gid]; target = int(ba_targets[gid]); n = g.num_nodes()
    pyg_data = dgl_to_pyg(g)
    node_wrapper = NodeModelWrapper(ba_model, target, pred_class); node_wrapper.eval()
    best_removed, best_size = None, float('inf')
    nodes_to_try = [target] + [i for i in range(min(n, 30)) if i != target]
    for node_idx in nodes_to_try:
        removed = greedy_edge_explanation(pyg_data, node_idx, node_wrapper, max_edges=10)
        if len(removed) > 0 and len(removed) < best_size:
            best_size = len(removed); best_removed = removed
            if best_size <= 4: break
    if best_removed is not None:
        ba_exp_dict[gid] = best_removed; ba_pred_dict[gid] = pred_class

print(f'Explained: {len(ba_exp_dict)}/{len(ba_target_gids)}')

# Stop the timer and print the result
end_time = time.perf_counter()
execution_time = end_time - start_time
print(f"Total execution time: {execution_time:.4f} seconds")

In [ ]:
# BA_Shapes metrics (argmax-based PN/PS, matching NodeExplainerEdgeMulti)
ba_pn = ba_ps = 0
ba_pres, ba_recs, ba_f1s, ba_accs, ba_exp_nums = [], [], [], [], []

for gid in ba_exp_dict:
    g = ba_graphs[gid]; removed = ba_exp_dict[gid]; orig_class = ba_pred_dict[gid]
    target = ba_targets[gid]; n = g.num_nodes()
    ba_exp_nums.append(len(removed) / 2.0)
    # PN
    pn_w = g.edata['weight'].clone().float()
    for (s, d) in removed: pn_w[s * n + d] = 0.0
    with torch.no_grad():
        pn_pred = ba_model(g, g.ndata['feat'].float(), pn_w, torch.tensor(target))
    if torch.argmax(pn_pred[0]).item() != orig_class: ba_pn += 1
    # PS
    ps_w = torch.zeros_like(g.edata['weight']).float()
    for (s, d) in removed: ps_w[s * n + d] = 1.0
    with torch.no_grad():
        ps_pred = ba_model(g, g.ndata['feat'].float(), ps_w, torch.tensor(target))
    if torch.argmax(ps_pred[0]).item() == orig_class: ba_ps += 1
    # Precision/Recall vs edata['gt']
    e_gts = g.edata['gt'].numpy(); exp_mask = np.zeros(n * n)
    for (s, d) in removed: exp_mask[s * n + d] = 1
    TP = sum(1 for i in range(n*n) if exp_mask[i]==1 and e_gts[i]==1)
    FP = sum(1 for i in range(n*n) if exp_mask[i]==1 and e_gts[i]!=1)
    FN = sum(1 for i in range(n*n) if exp_mask[i]!=1 and e_gts[i]==1)
    TN = n*n - TP - FP - FN
    pre = TP/(TP+FP) if TP > 0 else 0; rec = TP/(TP+FN) if TP > 0 else 0
    f1 = 2*pre*rec/(pre+rec) if (pre+rec) > 0 else 0; acc = (TP+TN)/(n*n)
    ba_pres.append(pre); ba_recs.append(rec); ba_f1s.append(f1); ba_accs.append(acc)

ba_total = len(ba_exp_dict)
ba_PN = ba_pn/ba_total; ba_PS = ba_ps/ba_total
ba_PNS = 2*ba_PN*ba_PS/(ba_PN+ba_PS) if (ba_PN+ba_PS) > 0 else 0
ba_avg = np.mean(ba_exp_nums)

print(f'average number of exps: {ba_avg:.2f}')
print(f'PN {ba_PN:.4f}'); print(f'PS {ba_PS:.4f}'); print(f'PNS {ba_PNS:.4f}')
print(f'acc:  {np.mean(ba_accs):.4f}  pre:  {np.mean(ba_pres):.4f}  '
      f'rec:  {np.mean(ba_recs):.4f}  f1:  {np.mean(ba_f1s):.4f}')

---
# 3. Tree_Cycles Dataset
- **Model**: GCNNodeTreeCycles (2-class node classification with sigmoid)
- **Selection**: Test nodes with correct non-zero class prediction
- **Metrics**: PN/PS use argmax-based flip (matching `NodeExplainerEdgeMulti`)

In [ ]:
# Load Tree_Cycles
TC_PATH = os.path.join(GNN_CFF_DIR, 'log', 'Tree_Cycles_logs')
tc_graphs, tc_label_dict = load_graphs(os.path.join(TC_PATH, 'dgl_graph.bin'))
tc_labels = tc_label_dict['labels']
tc_targets = np.load(os.path.join(TC_PATH, 'targets.pickle'), allow_pickle=True)
tc_test_indices = np.load(os.path.join(TC_PATH, 'test_indices.pickle'), allow_pickle=True)

tc_model = GCNNodeTreeCycles(tc_graphs[0].ndata['feat'].shape[1], 32,
                             num_classes=2, if_exp=True).to(DEVICE)
tc_model.load_state_dict(torch.load(os.path.join(TC_PATH, 'model.model'), map_location=DEVICE))
tc_model.eval()
for param in tc_model.parameters():
    param.requires_grad = False

correct = sum(1 for gid in tc_test_indices
              if torch.argmax(tc_model(tc_graphs[gid], tc_graphs[gid].ndata['feat'].float(),
                                       tc_graphs[gid].edata['weight'],
                                       torch.tensor(tc_targets[gid]))[0]).item()
              == torch.argmax(tc_labels[gid]).item())
print(f'Dataset: {len(tc_graphs)} subgraphs')
print(f'Test accuracy: {correct}/{len(tc_test_indices)} = {correct/len(tc_test_indices):.4f}')

In [ ]:
# Run greedy edge explanation on Tree_Cycles
tc_target_gids = []
for gid in tc_test_indices:
    g = tc_graphs[gid]; target = tc_targets[gid]
    with torch.no_grad():
        pred = tc_model(g, g.ndata['feat'].float(), g.edata['weight'], torch.tensor(target))
    pred_class = torch.argmax(pred[0]).item()
    true_class = torch.argmax(tc_labels[gid]).item()
    if pred_class != 0 and true_class != 0 and pred_class == true_class:
        tc_target_gids.append((gid, pred_class))
print(f'Target nodes: {len(tc_target_gids)}')

tc_exp_dict, tc_pred_dict = {}, {}
for gid, pred_class in tqdm.tqdm(tc_target_gids):
    g = tc_graphs[gid]; target = int(tc_targets[gid]); n = g.num_nodes()
    pyg_data = dgl_to_pyg(g)
    node_wrapper = NodeModelWrapper(tc_model, target, pred_class); node_wrapper.eval()
    best_removed, best_size = None, float('inf')
    for node_idx in range(n):
        removed = greedy_edge_explanation(pyg_data, node_idx, node_wrapper, max_edges=10)
        if len(removed) > 0 and len(removed) < best_size:
            best_size = len(removed); best_removed = removed
            if best_size <= 4: break
    if best_removed is not None:
        tc_exp_dict[gid] = best_removed; tc_pred_dict[gid] = pred_class

print(f'Explained: {len(tc_exp_dict)}/{len(tc_target_gids)}')

In [ ]:
# Tree_Cycles metrics
tc_pn = tc_ps = 0
tc_pres, tc_recs, tc_f1s, tc_accs, tc_exp_nums = [], [], [], [], []

for gid in tc_exp_dict:
    g = tc_graphs[gid]; removed = tc_exp_dict[gid]; orig_class = tc_pred_dict[gid]
    target = tc_targets[gid]; n = g.num_nodes()
    tc_exp_nums.append(len(removed) / 2.0)
    # PN
    pn_w = g.edata['weight'].clone().float()
    for (s, d) in removed: pn_w[s * n + d] = 0.0
    with torch.no_grad():
        pn_pred = tc_model(g, g.ndata['feat'].float(), pn_w, torch.tensor(target))
    if torch.argmax(pn_pred[0]).item() != orig_class: tc_pn += 1
    # PS
    ps_w = torch.zeros_like(g.edata['weight']).float()
    for (s, d) in removed: ps_w[s * n + d] = 1.0
    with torch.no_grad():
        ps_pred = tc_model(g, g.ndata['feat'].float(), ps_w, torch.tensor(target))
    if torch.argmax(ps_pred[0]).item() == orig_class: tc_ps += 1
    # Precision/Recall vs edata['gt']
    e_gts = g.edata['gt'].numpy(); exp_mask = np.zeros(n * n)
    for (s, d) in removed: exp_mask[s * n + d] = 1
    TP = sum(1 for i in range(n*n) if exp_mask[i]==1 and e_gts[i]==1)
    FP = sum(1 for i in range(n*n) if exp_mask[i]==1 and e_gts[i]!=1)
    FN = sum(1 for i in range(n*n) if exp_mask[i]!=1 and e_gts[i]==1)
    TN = n*n - TP - FP - FN
    pre = TP/(TP+FP) if TP > 0 else 0; rec = TP/(TP+FN) if TP > 0 else 0
    f1 = 2*pre*rec/(pre+rec) if (pre+rec) > 0 else 0; acc = (TP+TN)/(n*n)
    tc_pres.append(pre); tc_recs.append(rec); tc_f1s.append(f1); tc_accs.append(acc)

tc_total = len(tc_exp_dict)
tc_PN = tc_pn/tc_total; tc_PS = tc_ps/tc_total
tc_PNS = 2*tc_PN*tc_PS/(tc_PN+tc_PS) if (tc_PN+tc_PS) > 0 else 0
tc_avg = np.mean(tc_exp_nums)

print(f'average number of exps: {tc_avg:.2f}')
print(f'PN {tc_PN:.4f}'); print(f'PS {tc_PS:.4f}'); print(f'PNS {tc_PNS:.4f}')
print(f'acc:  {np.mean(tc_accs):.4f}  pre:  {np.mean(tc_pres):.4f}  '
      f'rec:  {np.mean(tc_recs):.4f}  f1:  {np.mean(tc_f1s):.4f}')

---
# Summary

In [ ]:
print('=' * 70)
print('  Greedy Edge Explanation Results (using gnn_cff pre-trained models)')
print('=' * 70)

datasets = [
    ('Mutagenicity_0', 'GCNGraph', len(exp_dict), len(target_gids),
     mutag_avg, mutag_PN, mutag_PS, mutag_PNS,
     np.mean(accs_list), np.mean(pres_list), np.mean(recs_list), np.mean(f1s_list)),
    ('BA_Shapes', 'GCNNodeBAShapes', len(ba_exp_dict), len(ba_target_gids),
     ba_avg, ba_PN, ba_PS, ba_PNS,
     np.mean(ba_accs), np.mean(ba_pres), np.mean(ba_recs), np.mean(ba_f1s)),
    ('Tree_Cycles', 'GCNNodeTreeCycles', len(tc_exp_dict), len(tc_target_gids),
     tc_avg, tc_PN, tc_PS, tc_PNS,
     np.mean(tc_accs), np.mean(tc_pres), np.mean(tc_recs), np.mean(tc_f1s)),
]

header = f'{"Dataset":<16} {"Model":<20} {"Exp":<8} {"avg_exp":>8} {"PN":>8} {"PS":>8} {"PNS":>8} {"Acc":>8} {"Pre":>8} {"Rec":>8} {"F1":>8}'
print(header)
print('-' * len(header))
for name, model, exp, total, avg, pn, ps, pns, acc, pre, rec, f1 in datasets:
    print(f'{name:<16} {model:<20} {exp}/{total:<5} {avg:>8.2f} {pn:>8.4f} {ps:>8.4f} {pns:>8.4f} {acc:>8.4f} {pre:>8.4f} {rec:>8.4f} {f1:>8.4f}')
print('=' * 70)